In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sohansakib75/yuyvvty")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/sohansakib75/yuyvvty...
Path to dataset files: /kaggle/input/datasets/sohansakib75/yuyvvty


In [2]:
import cv2
import numpy as np
import os
from PIL import Image
from tqdm import tqdm

base_dir = "/kaggle/input/datasets/sohansakib75/yuyvvty/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation"
classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
TARGET_SIZE = (224, 224)

output_base = "/kaggle/working/preprocessed_all"

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def alpha_trimmed_mean_filter(img, kernel_size=3, alpha=2):
    pad = kernel_size // 2
    padded = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    filtered = np.zeros_like(img)
    for i in range(pad, padded.shape[0] - pad):
        for j in range(pad, padded.shape[1] - pad):
            kernel = padded[i - pad:i + pad + 1, j - pad:j + pad + 1].flatten()
            kernel.sort()
            trimmed = kernel[alpha:-alpha]
            filtered[i - pad, j - pad] = np.mean(trimmed)
    return filtered

def zoom_roi_with_padding(img, bbox, zoom_factor=1.3):
    h, w = img.shape[:2]
    x, y, bw, bh = bbox
    
    # Calculate ROI center
    cx = x + bw // 2
    cy = y + bh // 2
    
    # Calculate new ROI size
    new_w = int(bw * zoom_factor)
    new_h = int(bh * zoom_factor)
    
    # Calculate new bounding box with padding around ROI center
    x1 = max(cx - new_w // 2, 0)
    y1 = max(cy - new_h // 2, 0)
    x2 = min(cx + new_w // 2, w)
    y2 = min(cy + new_h // 2, h)
    
    # Crop and zoom ROI area
    roi = img[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, (w, h), interpolation=cv2.INTER_LINEAR)
    
    return roi_resized

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize full image first
    resized = cv2.resize(original_rgb, TARGET_SIZE)

    # Convert to grayscale for detection
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)

    # Detect eyes
    eyes = eye_cascade.detectMultiScale(gray, 1.3, 5)
    if len(eyes) > 0:
        bbox = eyes[0]
    else:
        # Detect face
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) > 0:
            bbox = faces[0]
        else:
            # If no detection, use whole image
            bbox = (0, 0, resized.shape[1], resized.shape[0])

    # Zoom ROI with padding, keep image size
    zoomed_img = zoom_roi_with_padding(resized, bbox, zoom_factor=1.3)

    # Apply CLAHE on grayscale
    gray_zoomed = cv2.cvtColor(zoomed_img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray_zoomed)

    # Apply alpha trimmed mean filter
    filtered_img = alpha_trimmed_mean_filter(clahe_img)

    # Convert back to RGB
    processed_img = cv2.merge([filtered_img]*3)

    return processed_img

# Process all images, save results
for cls in classes:
    class_input_dir = os.path.join(base_dir, cls)
    class_output_dir = os.path.join(output_base, cls)
    os.makedirs(class_output_dir, exist_ok=True)

    image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for i, img_file in enumerate(tqdm(image_files, desc=f"Processing {cls}")):
        img_path = os.path.join(class_input_dir, img_file)
        preprocessed_img = preprocess_image(img_path)
        if preprocessed_img is not None:
            save_path = os.path.join(class_output_dir, f"image_{i+1}.jpg")
            Image.fromarray(preprocessed_img).save(save_path)

print("All images preprocessed and saved successfully!")


Processing Conjunctivitis: 100%|██████████| 357/357 [02:27<00:00,  2.42it/s]

All images preprocessed and saved successfully!


In [5]:
import cv2
import numpy as np
import os
from PIL import Image
import random

preprocessed_dir = "/kaggle/working/preprocessed_all"  # Folder where all preprocessed images saved
augmented_dir = "/kaggle/working/augmented_all"

TARGET_SIZE = (224, 224)
classes_3aug = ["Cataract", "Conjunctivitis", "Eyelid", "Normal"]
classes_4aug = ["Uveitis"]

def random_brightness(img):
    factor = 0.7 + 0.6 * random.random()  # brightness factor between 0.7-1.3
    img = cv2.convertScaleAbs(img, alpha=factor, beta=0)
    return img

def random_rotation(img, angle_range=15):
    angle = random.uniform(-angle_range, angle_range)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return rotated

def horizontal_flip(img):
    return cv2.flip(img, 1)

def random_shift_zoom(img, shift_ratio=0.1, zoom_range=0.1):
    h, w = img.shape[:2]
    max_dx = int(w * shift_ratio)
    max_dy = int(h * shift_ratio)
    dx = random.randint(-max_dx, max_dx)
    dy = random.randint(-max_dy, max_dy)
    M_shift = np.float32([[1, 0, dx], [0, 1, dy]])
    shifted = cv2.warpAffine(img, M_shift, (w, h), borderMode=cv2.BORDER_REPLICATE)
    zx = 1 + random.uniform(-zoom_range, zoom_range)
    M_zoom = cv2.getRotationMatrix2D((w/2, h/2), 0, zx)
    zoomed = cv2.warpAffine(shifted, M_zoom, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return zoomed

def augment_image(img):
    aug_funcs = [random_brightness, random_rotation, horizontal_flip, random_shift_zoom]
    np.random.shuffle(aug_funcs)
    img_aug = img.copy()
    for func in aug_funcs[:2]:
        img_aug = func(img_aug)
    return img_aug

def save_augmented_images(class_name, image_paths, n_augments):
    save_dir = os.path.join(augmented_dir, class_name)
    os.makedirs(save_dir, exist_ok=True)

    for idx, orig_img_path in enumerate(image_paths):
        img = cv2.imread(orig_img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Save original image
        orig_save_path = os.path.join(save_dir, f"image_{idx}_0.jpg")
        Image.fromarray(img_rgb).save(orig_save_path)

        # Save augmented images
        for i in range(1, n_augments + 1):
            aug_img = augment_image(img_rgb)
            aug_save_path = os.path.join(save_dir, f"image_{idx}_{i}.jpg")
            Image.fromarray(aug_img).save(aug_save_path)

# Run for all classes
for cls in classes_3aug:
    class_dir = os.path.join(preprocessed_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    save_augmented_images(cls, image_files, 3)

for cls in classes_4aug:
    class_dir = os.path.join(preprocessed_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    save_augmented_images(cls, image_files, 6)  # Augment 6 times for Uveitis

print(f"All images augmented and saved to {augmented_dir}/")


All images augmented and saved to /kaggle/working/augmented_all/


In [10]:
import os
import shutil
import random
from collections import defaultdict

# =========================================================
# PATHS
# =========================================================

SOURCE_DIR = "/kaggle/working/augmented_all"

OUTPUT_SPLIT_DIR = "/kaggle/working/bean_split_dataset"

TRAIN_DIR = os.path.join(OUTPUT_SPLIT_DIR, "train")
VAL_DIR   = os.path.join(OUTPUT_SPLIT_DIR, "val")
TEST_DIR  = os.path.join(OUTPUT_SPLIT_DIR, "test")

# =========================================================
# CREATE DIRECTORIES
# =========================================================

for directory in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(directory, exist_ok=True)

# =========================================================
# IMAGE EXTENSIONS
# =========================================================

image_extensions = (
    '.jpg',
    '.jpeg',
    '.png',
    '.bmp',
    '.tiff'
)

# =========================================================
# FIND CLASS FOLDERS
# =========================================================

class_folders = []

for root, dirs, files in os.walk(SOURCE_DIR):

    image_files = [
        f for f in files
        if f.lower().endswith(image_extensions)
    ]

    if len(image_files) > 0:
        class_folders.append(root)

# =========================================================
# SPLIT FUNCTION
# =========================================================

random.seed(42)

summary = []

print("\nStarting dataset split...\n")

for class_path in sorted(class_folders):

    class_name = os.path.basename(class_path)

    images = [
        os.path.join(class_path, f)
        for f in os.listdir(class_path)
        if f.lower().endswith(image_extensions)
    ]

    random.shuffle(images)

    total = len(images)

    # -----------------------------------------------------
    # 80 - 10 - 10 Split
    # -----------------------------------------------------

    train_count = int(total * 0.80)

    val_count = int(total * 0.10)

    test_count = total - train_count - val_count

    train_images = images[:train_count]

    val_images = images[
        train_count : train_count + val_count
    ]

    test_images = images[
        train_count + val_count :
    ]

    # =====================================================
    # COPY FILES
    # =====================================================

    split_map = {
        TRAIN_DIR: train_images,
        VAL_DIR: val_images,
        TEST_DIR: test_images
    }

    for split_dir, split_images in split_map.items():

        target_class_dir = os.path.join(
            split_dir,
            class_name
        )

        os.makedirs(
            target_class_dir,
            exist_ok=True
        )

        for img_path in split_images:

            shutil.copy(
                img_path,
                os.path.join(
                    target_class_dir,
                    os.path.basename(img_path)
                )
            )

    # =====================================================
    # SAVE SUMMARY
    # =====================================================

    summary.append({
        "Class": class_name,
        "Total": total,
        "Train": len(train_images),
        "Validation": len(val_images),
        "Test": len(test_images)
    })

print("\nDataset splitting completed!")

# =========================================================
# PRINT SUMMARY
# =========================================================

print("\n================ SPLIT SUMMARY ================\n")

total_train = 0
total_val = 0
total_test = 0
grand_total = 0

for row in summary:

    print(
        f"{row['Class']:<25}"
        f" Total: {row['Total']:<5}"
        f" Train: {row['Train']:<5}"
        f" Val: {row['Validation']:<5}"
        f" Test: {row['Test']:<5}"
    )

    total_train += row['Train']
    total_val += row['Validation']
    total_test += row['Test']
    grand_total += row['Total']

# =========================================================
# FINAL TOTALS
# =========================================================

print("\n================================================\n")

print(f"Grand Total Images : {grand_total}")

print(f"Training Images    : {total_train}")

print(f"Validation Images  : {total_val}")

print(f"Testing Images     : {total_test}")

print("\n================================================")

# =========================================================
# SAVE LOCATION
# =========================================================

print("\nDataset saved at:")

print(OUTPUT_SPLIT_DIR)


Starting dataset split...


Dataset splitting completed!

================ SPLIT SUMMARY ================

Cataract                  Total: 2176  Train: 1740  Val: 217   Test: 219  
Conjunctivitis            Total: 1428  Train: 1142  Val: 142   Test: 144  
Eyelid                    Total: 2100  Train: 1680  Val: 210   Test: 210  
Normal                    Total: 2596  Train: 2076  Val: 259   Test: 261  
Uveitis                   Total: 1561  Train: 1248  Val: 156   Test: 157  


Grand Total Images : 9861
Training Images    : 7886
Validation Images  : 984
Testing Images     : 991


Dataset saved at:
/kaggle/working/bean_split_dataset


In [12]:
import os
import copy
import random
import time
import subprocess
import numpy as np
import psutil
from PIL import ImageFilter

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as TF

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

try:
    from scipy.stats import binomtest
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False


# =========================================================
# REPRODUCIBILITY
# =========================================================

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# =========================================================
# PATHS
# =========================================================

OUTPUT_SPLIT_DIR = "/kaggle/working/bean_split_dataset"

TRAIN_DIR = os.path.join(OUTPUT_SPLIT_DIR, "train")
VAL_DIR   = os.path.join(OUTPUT_SPLIT_DIR, "val")
TEST_DIR  = os.path.join(OUTPUT_SPLIT_DIR, "test")

BEST_MODEL_PATH = "/kaggle/working/best_shufflenetv2_mobilenetv2_fusion.pth"


# =========================================================
# SETTINGS
# =========================================================

img_size = 224
batch_size = 32
num_epochs = 30

patience = 5
best_val_loss = float("inf")
epochs_no_improve = 0

learning_rate = 1e-4
weight_decay = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

print("Using device:", device)


# =========================================================
# GPU AND RAM USAGE FUNCTION
# =========================================================

def print_resource_usage(stage=""):
    print("\n================ RESOURCE USAGE ================")
    if stage:
        print(f"Stage: {stage}")

    ram = psutil.virtual_memory()
    print(f"RAM Used     : {ram.used / (1024 ** 3):.2f} GB")
    print(f"RAM Total    : {ram.total / (1024 ** 3):.2f} GB")
    print(f"RAM Usage    : {ram.percent:.2f}%")

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)

        allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
        max_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)

        print(f"GPU Name     : {gpu_name}")
        print(f"GPU Allocated: {allocated:.2f} GB")
        print(f"GPU Reserved : {reserved:.2f} GB")
        print(f"GPU Max Used : {max_allocated:.2f} GB")

        try:
            result = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used,memory.total",
                    "--format=csv,noheader,nounits"
                ],
                encoding="utf-8"
            )

            gpu_util, mem_used, mem_total = result.strip().split(", ")
            print(f"GPU Usage    : {gpu_util}%")
            print(f"GPU Memory   : {mem_used} MB / {mem_total} MB")

        except Exception:
            print("GPU Usage    : Could not read nvidia-smi output.")
    else:
        print("GPU Usage    : CUDA GPU not available.")

    print("================================================\n")


print_resource_usage("Before Training")


# =========================================================
# DATA TRANSFORMS
# =========================================================

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# DATASETS AND LOADERS
# =========================================================

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))

pin_memory = True if device.type == "cuda" else False

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory
)


# =========================================================
# FEATURE FUSION MODEL: MOBILENETV2 + SHUFFLENETV2
# =========================================================

def load_mobilenetv2():
    try:
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
    except Exception:
        model = models.mobilenet_v2(weights=None)
    return model


def load_shufflenetv2():
    try:
        weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
        model = models.shufflenet_v2_x1_0(weights=weights)
    except Exception:
        model = models.shufflenet_v2_x1_0(weights=None)
    return model


class ShuffleMobileFusionNet(nn.Module):
    def __init__(self, num_classes):
        super(ShuffleMobileFusionNet, self).__init__()

        self.mobilenet = load_mobilenetv2()
        self.shufflenet = load_shufflenetv2()

        self.mobile_features = self.mobilenet.features

        self.shuffle_conv1 = self.shufflenet.conv1
        self.shuffle_maxpool = self.shufflenet.maxpool
        self.shuffle_stage2 = self.shufflenet.stage2
        self.shuffle_stage3 = self.shufflenet.stage3
        self.shuffle_stage4 = self.shufflenet.stage4
        self.shuffle_conv5 = self.shufflenet.conv5

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        mobile_dim = 1280
        shuffle_dim = 1024
        fusion_dim = mobile_dim + shuffle_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        mobile_feat = self.mobile_features(x)
        mobile_feat = self.pool(mobile_feat)
        mobile_feat = torch.flatten(mobile_feat, 1)

        shuffle_feat = self.shuffle_conv1(x)
        shuffle_feat = self.shuffle_maxpool(shuffle_feat)
        shuffle_feat = self.shuffle_stage2(shuffle_feat)
        shuffle_feat = self.shuffle_stage3(shuffle_feat)
        shuffle_feat = self.shuffle_stage4(shuffle_feat)
        shuffle_feat = self.shuffle_conv5(shuffle_feat)
        shuffle_feat = self.pool(shuffle_feat)
        shuffle_feat = torch.flatten(shuffle_feat, 1)

        fused_feat = torch.cat([mobile_feat, shuffle_feat], dim=1)

        output = self.classifier(fused_feat)

        return output


model = ShuffleMobileFusionNet(num_classes=num_classes).to(device)


# =========================================================
# LOSS, OPTIMIZER, SCHEDULER
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


# =========================================================
# TRAINING AND VALIDATION FUNCTIONS
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# =========================================================
# MODEL TRAINING WITH EARLY STOPPING
# =========================================================

best_model_wts = copy.deepcopy(model.state_dict())

training_start_time = time.time()

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        device
    )

    val_loss, val_acc = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"LR: {current_lr:.8f}"
    )

    print_resource_usage(f"After Epoch {epoch + 1}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(model.state_dict())

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best model saved.")

    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= patience:
        print("Early stopping activated.")
        break


training_end_time = time.time()
training_time_minutes = (training_end_time - training_start_time) / 60

print(f"\nTotal Training Time: {training_time_minutes:.2f} minutes")

model.load_state_dict(best_model_wts)

print_resource_usage("After Training")


# =========================================================
# FINAL TEST PREDICTION FUNCTION
# =========================================================

def get_predictions(model, loader, criterion, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    running_loss = 0.0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * images.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    test_loss = running_loss / total

    return (
        test_loss,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs)
    )


# =========================================================
# STATISTICAL TESTS AND METRICS
# =========================================================

def bootstrap_accuracy_ci(y_true, y_pred, n_bootstrap=1000, confidence=0.95, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    n = len(y_true)
    bootstrap_scores = []

    for _ in range(n_bootstrap):
        indices = rng.choice(np.arange(n), size=n, replace=True)
        score = accuracy_score(y_true[indices], y_pred[indices])
        bootstrap_scores.append(score)

    lower_percentile = ((1.0 - confidence) / 2.0) * 100
    upper_percentile = (confidence + ((1.0 - confidence) / 2.0)) * 100

    lower = np.percentile(bootstrap_scores, lower_percentile)
    upper = np.percentile(bootstrap_scores, upper_percentile)

    return lower, upper


def print_statistical_results(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    precision_macro = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    ci_lower, ci_upper = bootstrap_accuracy_ci(
        y_true,
        y_pred,
        n_bootstrap=1000,
        confidence=0.95,
        seed=42
    )

    correct_predictions = int(np.sum(y_true == y_pred))
    total_predictions = len(y_true)
    chance_level = 1.0 / len(class_names)

    print("\n================ FINAL FUSION MODEL STATISTICAL RESULTS ================\n")

    print(f"Total Test Samples              : {total_predictions}")
    print(f"Correct Predictions             : {correct_predictions}")
    print(f"Accuracy                        : {acc * 100:.2f}%")
    print(f"Balanced Accuracy               : {balanced_acc * 100:.2f}%")
    print(f"Macro Precision                 : {precision_macro * 100:.2f}%")
    print(f"Macro Recall                    : {recall_macro * 100:.2f}%")
    print(f"Macro F1-score                  : {f1_macro * 100:.2f}%")
    print(f"Weighted F1-score               : {f1_weighted * 100:.2f}%")
    print(f"Cohen's Kappa Score             : {kappa:.4f}")
    print(f"Matthews Correlation Coefficient: {mcc:.4f}")
    print(f"95% Bootstrap Accuracy CI       : [{ci_lower * 100:.2f}%, {ci_upper * 100:.2f}%]")

    if SCIPY_AVAILABLE:
        binom_result = binomtest(
            correct_predictions,
            total_predictions,
            p=chance_level,
            alternative="greater"
        )

        print(f"Chance Level                    : {chance_level * 100:.2f}%")
        print(f"Exact Binomial Test Statistic   : {correct_predictions}/{total_predictions}")
        print(f"Exact Binomial Test p-value     : {binom_result.pvalue:.6e}")
    else:
        print("Exact Binomial Test             : scipy not available.")

    print("\n================ CONFUSION MATRIX ================\n")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)

    print("\n================ CLASSIFICATION REPORT ================\n")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            zero_division=0
        )
    )


# =========================================================
# FINAL TEST EVALUATION
# =========================================================

test_loss, y_true, y_pred, y_probs = get_predictions(
    model,
    test_loader,
    criterion,
    device
)

test_acc = accuracy_score(y_true, y_pred)

print("\n================ FINAL TEST PERFORMANCE ================\n")
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

print_statistical_results(
    y_true,
    y_pred,
    class_names
)

print_resource_usage("After Final Test Evaluation")


# =========================================================
# PERTURBATION CLASSES
# NOTE:
# Gaussian Noise and Low Contrast were removed as requested.
# =========================================================

class Cutout:
    def __init__(self, size=56):
        self.size = size

    def __call__(self, tensor):
        _, h, w = tensor.shape

        y = random.randint(0, h - 1)
        x = random.randint(0, w - 1)

        y1 = max(0, y - self.size // 2)
        y2 = min(h, y + self.size // 2)
        x1 = max(0, x - self.size // 2)
        x2 = min(w, x + self.size // 2)

        tensor[:, y1:y2, x1:x2] = 0.0

        return tensor


# =========================================================
# PERTURBED TEST TRANSFORMS
# Gaussian Noise and Low Contrast transforms removed.
# =========================================================

low_light_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Lambda(lambda img: TF.adjust_brightness(img, 0.45)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

high_light_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Lambda(lambda img: TF.adjust_brightness(img, 1.65)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

blur_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Lambda(lambda img: img.filter(ImageFilter.GaussianBlur(radius=2))),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

cutout_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    Cutout(size=56),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# STANDARD PERTURBED ACCURACY FUNCTION
# =========================================================

def evaluate_accuracy(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += torch.sum(preds == labels).item()
            total += labels.size(0)

    return correct / total


# =========================================================
# CUTMIX PERTURBATION EVALUATION
# =========================================================

def rand_bbox(size, lam):
    batch_size, channels, height, width = size

    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(width * cut_ratio)
    cut_h = int(height * cut_ratio)

    cx = np.random.randint(width)
    cy = np.random.randint(height)

    bbx1 = np.clip(cx - cut_w // 2, 0, width)
    bby1 = np.clip(cy - cut_h // 2, 0, height)
    bbx2 = np.clip(cx + cut_w // 2, 0, width)
    bby2 = np.clip(cy + cut_h // 2, 0, height)

    return bbx1, bby1, bbx2, bby2


def evaluate_cutmix_accuracy(model, loader, device, alpha=1.0):
    model.eval()

    total_correct = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            batch_size = images.size(0)

            lam = np.random.beta(alpha, alpha)
            rand_index = torch.randperm(batch_size).to(device)

            labels_a = labels
            labels_b = labels[rand_index]

            bbx1, bby1, bbx2, bby2 = rand_bbox(images.size(), lam)

            images[:, :, bby1:bby2, bbx1:bbx2] = images[
                rand_index,
                :,
                bby1:bby2,
                bbx1:bbx2
            ]

            lam = 1.0 - (
                (bbx2 - bbx1) * (bby2 - bby1)
                / (images.size(-1) * images.size(-2))
            )

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct_a = preds.eq(labels_a).float()
            correct_b = preds.eq(labels_b).float()

            weighted_correct = lam * correct_a + (1.0 - lam) * correct_b

            total_correct += weighted_correct.sum().item()
            total_samples += batch_size

    return total_correct / total_samples


# =========================================================
# CREATE PERTURBED TEST LOADERS
# Gaussian Noise and Low Contrast removed from this dictionary.
# =========================================================

perturbation_transforms = {
    "Low Lighting Accuracy": low_light_transform,
    "High Lighting Accuracy": high_light_transform,
    "Blur Accuracy": blur_transform,
    "Cutout Accuracy": cutout_transform
}

perturbed_results = {}

for perturb_name, perturb_transform in perturbation_transforms.items():

    perturbed_dataset = datasets.ImageFolder(
        TEST_DIR,
        transform=perturb_transform
    )

    perturbed_loader = DataLoader(
        perturbed_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=pin_memory
    )

    acc = evaluate_accuracy(
        model,
        perturbed_loader,
        device
    )

    perturbed_results[perturb_name] = acc

    print_resource_usage(f"After {perturb_name}")


# CutMix uses the normal test transform, then mixes image regions batch-wise.

cutmix_acc = evaluate_cutmix_accuracy(
    model,
    test_loader,
    device,
    alpha=1.0
)

perturbed_results["CutMix Accuracy"] = cutmix_acc


# =========================================================
# PRINT PERTURBED ACCURACIES
# =========================================================

print("\n================ PERTURBED ROBUSTNESS ACCURACY ================\n")

for name, acc in perturbed_results.items():
    print(f"{name:<30}: {acc * 100:.2f}%")

print("\nBest model saved at:")
print(BEST_MODEL_PATH)

print_resource_usage("End of Full Pipeline")

Using device: cuda

================ RESOURCE USAGE ================
Stage: Before Training
RAM Used     : 2.62 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.50%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

Classes: ['Cataract', 'Conjunctivitis', 'Eyelid', 'Normal', 'Uveitis']
Number of classes: 5
Training images: 7886
Validation images: 984
Testing images: 991


/tmp/ipykernel_57/574432698.py:306: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_57/574432698.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [1/30] Train Loss: 0.8625 Train Acc: 0.6787 Val Loss: 0.4659 Val Acc: 0.8506 LR: 0.00010000

================ RESOURCE USAGE ================
Stage: After Epoch 1
RAM Used     : 2.65 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.60%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

Best model saved.


/tmp/ipykernel_57/574432698.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [2/30] Train Loss: 0.3917 Train Acc: 0.8767 Val Loss: 0.2835 Val Acc: 0.9126 LR: 0.00010000

================ RESOURCE USAGE ================
Stage: After Epoch 2
RAM Used     : 2.66 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.60%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

Best model saved.


...

Epoch [25/30] Train Loss: 0.0079 Train Acc: 0.9986 Val Loss: 0.0101 Val Acc: 0.9949 LR: 0.00000900

================ RESOURCE USAGE ================
Stage: After Epoch 25
RAM Used     : 2.62 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.50%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

No improvement for 3 epoch(s).


/tmp/ipykernel_57/574432698.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [26/30] Train Loss: 0.0049 Train Acc: 0.9994 Val Loss: 0.0105 Val Acc: 0.9970 LR: 0.00000900

================ RESOURCE USAGE ================
Stage: After Epoch 26
RAM Used     : 2.65 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.60%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

No improvement for 4 epoch(s).


/tmp/ipykernel_57/574432698.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [27/30] Train Loss: 0.0040 Train Acc: 0.9991 Val Loss: 0.0108 Val Acc: 0.9980 LR: 0.00000900

================ RESOURCE USAGE ================
Stage: After Epoch 27
RAM Used     : 2.63 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.50%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.

No improvement for 5 epoch(s).
Early stopping activated.

Total Training Time: 11.57 minutes

================ RESOURCE USAGE ================
Stage: After Training
RAM Used     : 2.63 GB
RAM Total    : 31.35 GB
RAM Usage    : 10.50%
GPU Name     : Tesla T4
GPU Allocated: 0.20 GB
GPU Reserved : 1.84 GB
GPU Max Used : 3.08 GB
GPU Usage    : Could not read nvidia-smi output.


================ FINAL TEST PERFORMANCE ================

Test Loss    : 0.0033
Test Accuracy: 100.00%

================ FINAL FUSION MODEL STATISTICAL RESULTS ================

Total Test Samples              : 991
Correct Predictions    